# Cell 1 — Setup + Load/Freeze Baseline ViT

In [1]:
# Cell 1 — Setup + Load/Freeze Baseline ViT

# (A) Installs (idempotent)
!pip -q install timm==0.9.10

# (B) Imports & seeds
import os, json, glob, random
import numpy as np
import torch, timm
import torch.nn as nn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# (C) Auto-locate inputs (you can override manually if needed)
# It will search /kaggle/input/** for vit_cls_best.pt & meta_cls.json
def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if len(hits) > 0 else None

CKPT_PATH  = find_one("/kaggle/input/**/vit_cls_best.pt")    # <- your fine-tuned weights
META_PATH  = find_one("/kaggle/input/**/meta_cls.json")       # <- model meta
LABELS_PATH= find_one("/kaggle/input/**/labels.txt")          # <- id2label

print("Found checkpoint :", CKPT_PATH)
print("Found meta       :", META_PATH)
print("Found labels     :", LABELS_PATH)

assert CKPT_PATH and META_PATH and LABELS_PATH, "Missing one of {checkpoint, meta_cls.json, labels.txt}"

# (D) Read meta/labels
with open(META_PATH, "r") as f:
    meta = json.load(f)

with open(LABELS_PATH, "r") as f:
    id2label = [ln.strip() for ln in f if ln.strip()]

print("timm_name:", meta["timm_name"], "| num_classes:", meta["num_classes"])

# (E) Rebuild model skeleton from meta and load weights
model = timm.create_model(
    meta["timm_name"],
    pretrained=False,
    num_classes=meta["num_classes"],
)
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.to(DEVICE)
model.eval()

# (F) Freeze EVERYTHING (Phase-1: pure CFE)
for p in model.parameters():
    p.requires_grad = False

# (G) Discover CLS dim (D) for ViT-B/16 it should be 768
# timm ViT exposes forward_features; CLS token = [:, 0, :]
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
    feats = model.forward_features(dummy)     # [1, tokens, D] typically
    if isinstance(feats, torch.Tensor) and feats.ndim == 3:
        D = feats.shape[-1]
        cls = feats[:, 0]                     # [1, D]
    else:
        # some timm variants return dict; try 'pre_logits' or 'x_norm_clstoken'
        try:
            x = feats["x_norm_clstoken"]
            D = x.shape[-1]; cls = x
        except Exception:
            raise RuntimeError("Couldn't read CLS embedding from forward_features output.")

    logits = model.forward_head(feats, pre_logits=False)  # [1, num_classes]

print(f"CLS dim (D): {D}")
print(f"logits shape: {tuple(logits.shape)} (expect [1, {meta['num_classes']}])")

# (H) Keep a handle to classifier head (Linear  D -> num_classes)
# Many timm ViTs name it 'head'; fall back to 'fc' if needed.
head = getattr(model, "head", None) or getattr(model, "fc", None)
assert isinstance(head, nn.Module), "Couldn't find classifier head (head/fc)."
# Freeze head as well for Phase-1 (can unfreeze later if we want)
for p in head.parameters():
    p.requires_grad = False

print(head)
print("Baseline ViT loaded & frozen. Ready for CFE layers in next cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Cell 2 — MaskGenerator + CFWrapper (PP-ready)

In [2]:
# Cell 2 — MaskGenerator + CFWrapper (PP-ready)

import torch
import torch.nn as nn

# --- 1) MaskGenerator: CLS [B, D] -> fmask [B, D]
#     mode='PP' => Sigmoid in [0,1]
#     mode='PN' => ReLU >=0   (we'll start with PP)
class MaskGenerator(nn.Module):
    def __init__(self, d: int, mode: str = "PP"):
        super().__init__()
        assert mode in ("PP", "PN")
        self.mode = mode
        self.net = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(inplace=True),
            nn.Linear(d, d),
            nn.Sigmoid() if mode == "PP" else nn.ReLU(inplace=True),
        )

    def forward(self, cls: torch.Tensor) -> torch.Tensor:
        # cls: [B, D]
        return self.net(cls)


# --- 2) Helper: get CLS token robustly from timm ViT
def get_cls_from_features(features: torch.Tensor | dict) -> torch.Tensor:
    """
    Returns CLS embedding [B, D] from timm ViT forward_features output.
    Works for the common tensor-returning ViTs; has a dict fallback.
    """
    if isinstance(features, torch.Tensor) and features.ndim == 3:
        return features[:, 0]  # [B, tokens, D] -> CLS [B, D]
    if isinstance(features, dict):
        # common key in many timm vit impls
        if "x_norm_clstoken" in features:
            return features["x_norm_clstoken"]  # [B, D]
        if "pre_logits" in features:
            x = features["pre_logits"]
            return x if x.ndim == 2 else x[:, 0]
    raise RuntimeError("Could not extract CLS embedding from features.")


# --- 3) CFWrapper: frozen ViT + frozen head + trainable G
class CFWrapper(nn.Module):
    def __init__(self, vit: nn.Module, head: nn.Module, d: int, mode: str = "PP"):
        super().__init__()
        assert mode in ("PP", "PN")
        self.vit = vit.eval()        # keep frozen/eval
        self.head = head.eval()      # keep frozen/eval
        self.G = MaskGenerator(d=d, mode=mode)
        self.mode = mode
        self.d = d

        # Ensure vit & head are frozen (safety)
        for p in self.vit.parameters():  p.requires_grad = False
        for p in self.head.parameters(): p.requires_grad = False

    @torch.no_grad()
    def _forward_cls(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.vit.forward_features(x)
        cls = get_cls_from_features(feats)  # [B, D]
        return cls

    def forward(self, x: torch.Tensor, threshold_test: bool = False):
        """
        Returns:
          probs     : [B, C]
          logits    : [B, C]
          fmask     : [B, D]
          cls       : [B, D] (frozen features)
          modified  : [B, D] (masked features)
        """
        # 1) Frozen CLS features
        with torch.no_grad():
            cls = self._forward_cls(x)   # [B, D]

        # 2) Trainable mask
        fmask = self.G(cls)              # [B, D]
        if threshold_test and self.mode == "PP":
            fmask = (fmask > 0.5).float()

        # 3) Apply mask
        if self.mode == "PP":
            modified = cls * fmask
        else:  # PN
            modified = cls + fmask

        # 4) Logits via frozen head (same classifier as baseline)
        logits = self.head(modified)     # [B, C]
        probs  = torch.softmax(logits, dim=1)
        return probs, logits, fmask, cls, modified


# --- 4) Instantiate wrapper (PP mode) and quick sanity forward
D = 768  # from Cell-1 print; keep in sync if you change backbone
cf_pp = CFWrapper(model, head, d=D, mode="PP").to(DEVICE)

with torch.no_grad():
    xb = torch.randn(2, 3, 224, 224, device=DEVICE)
    probs, logits, fmask, cls, modified = cf_pp(xb)
    print("probs:", tuple(probs.shape), "logits:", tuple(logits.shape))
    print("cls:", tuple(cls.shape), "fmask:", tuple(fmask.shape), "modified:", tuple(modified.shape))

print("CFWrapper (PP) ready. Next cell: losses + optimizer (only G params).")

probs: (2, 200) logits: (2, 200)
cls: (2, 768) fmask: (2, 768) modified: (2, 768)
CFWrapper (PP) ready. Next cell: losses + optimizer (only G params).


# Parse CUB + Datasets + Dataloaders (official split + val=15% of train)

In [3]:
# Cell — Parse CUB + Datasets + Dataloaders (official split + val=15% of train)
# Dataset: https://www.kaggle.com/datasets/wenewone/cub2002011

import os, json, random
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Paths ----
DATA_ROOT = "/kaggle/input/cub2002011/CUB_200_2011"
IMAGES_DIR = os.path.join(DATA_ROOT, "images")
FILES = {
    "classes": os.path.join(DATA_ROOT, "classes.txt"),
    "images": os.path.join(DATA_ROOT, "images.txt"),
    "labels": os.path.join(DATA_ROOT, "image_class_labels.txt"),
    "split" : os.path.join(DATA_ROOT, "train_test_split.txt"),
}
for k,v in FILES.items():
    assert os.path.isfile(v), f"Missing file: {k} -> {v}"
assert os.path.isdir(IMAGES_DIR), f"Missing images dir: {IMAGES_DIR}"

# ---- Helpers to read CUB metadata ----
def read_kv_int_str(path):
    out = {}
    with open(path, "r") as f:
        for ln in f:
            ln = ln.strip()
            if not ln: 
                continue
            a,b = ln.split(" ", 1)
            out[int(a)] = b.strip()
    return out

def read_kv_int_int(path):
    out = {}
    with open(path, "r") as f:
        for ln in f:
            ln = ln.strip()
            if not ln: 
                continue
            a,b = ln.split()
            out[int(a)] = int(b)
    return out

# ---- Load meta ----
classes_map = read_kv_int_str(FILES["classes"])         # 1..200 -> name
images_map  = read_kv_int_str(FILES["images"])          # img_id -> rel path
labels_map  = read_kv_int_int(FILES["labels"])          # img_id -> 1..200
split_map   = read_kv_int_int(FILES["split"])           # img_id -> 1(train) / 0(test)

n_classes = len(classes_map)
assert n_classes == 200, f"Expected 200 classes, got {n_classes}"

# ---- Build full dataframe ----
rows = []
for img_id, rel_path in images_map.items():
    class_id_1b = labels_map[img_id]
    class_id_0b = class_id_1b - 1
    class_name  = classes_map[class_id_1b]
    is_train    = split_map[img_id] == 1
    abs_path    = os.path.join(IMAGES_DIR, rel_path)
    rows.append({
        "img_id": img_id,
        "rel_path": rel_path,
        "abs_path": abs_path,
        "class_id": class_id_0b,
        "class_name": class_name.replace("_", " "),
        "split": "train" if is_train else "test",
    })
df_all = pd.DataFrame(rows)

# ---- Official train/test; make val from train (stratified 15%) ----
df_train = df_all[df_all["split"] == "train"].copy()
df_test  = df_all[df_all["split"] == "test"].copy()

train_idx, val_idx = train_test_split(
    np.arange(len(df_train)),
    test_size=0.15,
    random_state=SEED,
    stratify=df_train["class_id"].values
)
df_train_final = df_train.iloc[train_idx].reset_index(drop=True)
df_val_final   = df_train.iloc[val_idx].reset_index(drop=True)

# ---- Save CSVs (useful for reproducibility / downstream analysis) ----
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
train_csv = os.path.join(OUTPUT_DIR, "cub_train.csv")
val_csv   = os.path.join(OUTPUT_DIR, "cub_val.csv")
test_csv  = os.path.join(OUTPUT_DIR, "cub_test.csv")
df_train_final.to_csv(train_csv, index=False)
df_val_final.to_csv(val_csv, index=False)
df_test.to_csv(test_csv, index=False)

# ---- Transforms (ViT-friendly) ----
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ---- Dataset / Dataloaders ----
class CUBDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["abs_path"].tolist()
        self.labels = df["class_id"].astype(int).tolist()
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]; y = self.labels[idx]
        img = Image.open(p).convert("RGB")
        img = self.transform(img)
        return img, y

ds_train = CUBDataset(df_train_final, train_tf)
ds_val   = CUBDataset(df_val_final,   eval_tf)
ds_test  = CUBDataset(df_test,        eval_tf)

BATCH_SIZE  = 32
NUM_WORKERS = 2

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

# ---- Sanity check ----
xb, yb = next(iter(dl_train))
print(f"Train batch: x={tuple(xb.shape)}, y={tuple(yb.shape)}, dtype={xb.dtype}, device={xb.device}")
print(f"Classes present — train: {len(set(ds_train.labels))} | val: {len(set(ds_val.labels))} | test: {len(set(ds_test.labels))}")
print("CSV saved to:", train_csv, "|", val_csv, "|", test_csv)

Train batch: x=(32, 3, 224, 224), y=(32,), dtype=torch.float32, device=cpu
Classes present — train: 200 | val: 200 | test: 200
CSV saved to: /kaggle/working/cub_train.csv | /kaggle/working/cub_val.csv | /kaggle/working/cub_test.csv


# Cell 3 — CFE (PP) : CE + L1 losses, optimizer, training loop

In [4]:
# Cell 3 — CFE (PP) : CE + L1 losses, optimizer, training loop

import os, json, time
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ---- Config (tune if needed) ----
@dataclass
class CFEConfig:
    mode: str = "PP"          # "PP" or "PN" (we start with PP)
    lambda_l1: float = 1e-3   # sparsity weight (PP ~1e-3)
    weight_ce: float = 1.0    # CE multiplier
    lr: float = 1e-4          # AdamW LR
    epochs: int = 50
    patience: int = 5         # early stop on val CE
    save_dir: str = "/kaggle/working"

CFG = CFEConfig()

# ---- CE + L1 loss (no pre-softmax bonus in Phase-1) ----
_ce = nn.CrossEntropyLoss()
def cfe_loss(logits, fmask, alter_targets, weight_ce=1.0, lambda_l1=1e-3):
    ce = _ce(logits, alter_targets)
    l1 = fmask.abs().sum(dim=1).mean()
    loss = weight_ce * ce + lambda_l1 * l1
    return loss, ce.detach(), l1.detach()

# ---- Optimizer: only G (mask generator) is trainable ----
optimG = torch.optim.AdamW(cf_pp.G.parameters(), lr=CFG.lr, weight_decay=0.0)
# (SGD option)
# optimG = torch.optim.SGD(cf_pp.G.parameters(), lr=1e-2, momentum=0.9)

# ---- One-epoch helper ----
def run_epoch(loader: DataLoader, train: bool, alter_class_id: int):
    cf_pp.train(mode=train)  # vit/head stay eval; only G respects train/eval
    total, total_ce, total_l1, total_count = 0.0, 0.0, 0.0, 0
    nz_ratio_accum = 0.0

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)      # not used directly
        alter = torch.full_like(yb, alter_class_id, dtype=torch.long)

        if train:
            optimG.zero_grad(set_to_none=True)

        probs, logits, fmask, cls, modified = cf_pp(xb)
        loss, ce_val, l1_val = cfe_loss(
            logits, fmask, alter,
            weight_ce=CFG.weight_ce, lambda_l1=CFG.lambda_l1
        )

        if train:
            loss.backward()
            optimG.step()

        bs = xb.size(0)
        total      += loss.item() * bs
        total_ce   += ce_val.item() * bs
        total_l1   += l1_val.item() * bs
        total_count += bs

        # sparsity indicator: fraction of nonzero entries (post-activation)
        nz_ratio_accum += (fmask.abs() > 1e-6).float().mean().item() * bs

    avg = total / max(1, total_count)
    avg_ce = total_ce / max(1, total_count)
    avg_l1 = total_l1 / max(1, total_count)
    avg_nz = nz_ratio_accum / max(1, total_count)
    return {"loss": avg, "ce": avg_ce, "l1": avg_l1, "nz_ratio": avg_nz}

# ---- Train CFE for ONE alter class (fixed) ----
def train_cfe_for_class(alter_class_id: int, run_name: str | None = None):
    start = time.time()
    best_val_ce = float("inf")
    no_improve = 0

    name = run_name or f"G_{CFG.mode}_class_{alter_class_id:03d}"
    os.makedirs(CFG.save_dir, exist_ok=True)
    save_path = os.path.join(CFG.save_dir, f"{name}.pt")
    meta_path = os.path.join(CFG.save_dir, f"{name}.json")

    print(f"==> Train CFE (mode={CFG.mode}) for alter_class={alter_class_id}")
    print(f"    lambda_l1={CFG.lambda_l1} | lr={CFG.lr} | epochs={CFG.epochs}")

    for ep in range(1, CFG.epochs + 1):
        tr = run_epoch(dl_train, train=True,  alter_class_id=alter_class_id)
        va = run_epoch(dl_val,   train=False, alter_class_id=alter_class_id)

        print(f"[{ep:02d}/{CFG.epochs}] "
              f"train: loss {tr['loss']:.4f} (ce {tr['ce']:.4f} | l1 {tr['l1']:.4f} | nz {tr['nz_ratio']:.3f}) | "
              f"val: loss {va['loss']:.4f} (ce {va['ce']:.4f} | l1 {va['l1']:.4f} | nz {va['nz_ratio']:.3f})")

        # Early stop on validation CE
        if va["ce"] + 1e-9 < best_val_ce:
            best_val_ce = va["ce"]
            no_improve = 0
            torch.save(cf_pp.G.state_dict(), save_path)
            with open(meta_path, "w") as f:
                json.dump({
                    "mode": CFG.mode,
                    "alter_class_id": alter_class_id,
                    "lambda_l1": CFG.lambda_l1,
                    "weight_ce": CFG.weight_ce,
                    "lr": CFG.lr,
                    "epochs": CFG.epochs,
                    "D": cf_pp.d,
                    "vit_frozen": True,
                    "head_frozen": True
                }, f, indent=2)
            print(f"  ↳ saved best G to {save_path}")
        else:
            no_improve += 1
            if no_improve >= CFG.patience:
                print(f"Early stopping (no val CE improvement for {CFG.patience} epochs).")
                break

    elapsed = time.time() - start
    print(f"Done in {elapsed/60:.1f} min. Best val CE: {best_val_ce:.4f}")
    print("Generator path:", save_path)
    return save_path, meta_path

# ---- Convenience: show a class name from labels.txt/meta (optional) ----
# id2label already loaded in Cell-1; if not, set id2label = [..list..]
def show_class(idx: int):
    name = id2label[idx] if 0 <= idx < len(id2label) else str(idx)
    print(f"{idx}: {name}")

# Cell 4 — Train CFE (PP) on bottom-5 classes only + save all artifacts

In [5]:
# Cell 4 — Train CFE (PP) on bottom-5 classes only + save all artifacts

import os, json, time, zipfile
import numpy as np
import pandas as pd

# Where to drop all per-class generators
SAVE_ROOT = os.path.join(CFG.save_dir, "cfe_all")
os.makedirs(SAVE_ROOT, exist_ok=True)

def pick_top5_from_csv(csv_path="/kaggle/input/vit_classifier/pytorch/default/1/per_class_accuracy.csv"):
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        # expected columns: class_id, class_name, total, correct, acc
        df = df.sort_values("acc", ascending=False)
        ids = df["class_id"].astype(int).tolist()[:5]
        print("Picked top-5 by accuracy:", ids, [id2label[i] for i in ids])
        return ids
    else:
        print("per_class_accuracy.csv not found — falling back to first 5 IDs (pilot).")
        return list(range(5))

def train_one_class(alter_id: int):
    # Point the generator save directory to cfe_all
    old_dir = CFG.save_dir
    CFG.save_dir = SAVE_ROOT
    try:
        name = f"G_PP_class_{alter_id:03d}"
        print("\n" + "="*60)
        print(f"Training CFE (PP) for class {alter_id:3d}: {id2label[alter_id]}")
        save_path, meta_path = train_cfe_for_class(alter_class_id=alter_id, run_name=name)
    finally:
        CFG.save_dir = old_dir
    return save_path, meta_path

def run_top5():
    focus_ids = pick_top5_from_csv()
    logs = []
    t0 = time.time()
    for cid in focus_ids:
        sp, mp = train_one_class(cid)
        logs.append({"class_id": cid, "class_name": id2label[cid], "gen_path": sp, "meta_path": mp})
    # save CSV log
    log_df = pd.DataFrame(logs)
    log_csv = os.path.join(SAVE_ROOT, "cfe_train_log.csv")
    log_df.to_csv(log_csv, index=False)
    print("\nSaved summary:", log_csv)

    # zip everything for easy download/reuse
    zip_path = os.path.join(CFG.save_dir, "cfe_all_top5.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(SAVE_ROOT):
            for f in files:
                fp = os.path.join(root, f)
                zf.write(fp, arcname=os.path.relpath(fp, SAVE_ROOT))
    print(f"Zipped artifacts → {zip_path}")
    print(f"All done in {(time.time()-t0)/60:.2f} min.")
    return focus_ids, zip_path

# ---- launch ----
focus, zip_file = run_top5()
print("Classes trained:", focus)

Picked top-5 by accuracy: [17, 73, 92, 69, 168] ['018.Spotted Catbird', '074.Florida Jay', '093.Clark Nutcracker', '070.Green Violetear', '169.Magnolia Warbler']

Training CFE (PP) for class  17: 018.Spotted Catbird
==> Train CFE (mode=PP) for alter_class=17
    lambda_l1=0.001 | lr=0.0001 | epochs=50
[01/50] train: loss 4.5771 (ce 4.3219 | l1 255.1626 | nz 1.000) | val: loss 3.7568 (ce 3.5376 | l1 219.1671 | nz 1.000)
  ↳ saved best G to /kaggle/working/cfe_all/G_PP_class_017.pt
[02/50] train: loss 3.4898 (ce 3.2653 | l1 224.5293 | nz 1.000) | val: loss 3.0430 (ce 2.8144 | l1 228.5543 | nz 1.000)
  ↳ saved best G to /kaggle/working/cfe_all/G_PP_class_017.pt
[03/50] train: loss 2.9414 (ce 2.7104 | l1 230.9872 | nz 0.998) | val: loss 2.6623 (ce 2.4291 | l1 233.2508 | nz 0.984)
  ↳ saved best G to /kaggle/working/cfe_all/G_PP_class_017.pt
[04/50] train: loss 2.6456 (ce 2.4121 | l1 233.5045 | nz 0.975) | val: loss 2.4459 (ce 2.2105 | l1 235.4830 | nz 0.922)
  ↳ saved best G to /kaggle/wor

# Cell 5 — Evaluate trained generators (PP) on test set

In [6]:
# Cell 4 — Evaluate trained generators (PP) and save artifacts (FIXED)

import os, re, json, glob, zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- use the same MaskGenerator we trained ----
class MaskGenerator(torch.nn.Module):
    def __init__(self, d: int, mode: str = "PP"):
        super().__init__()
        assert mode in ("PP", "PN")
        self.mode = mode
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d, d),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(d, d),
            torch.nn.Sigmoid() if mode == "PP" else torch.nn.ReLU(inplace=True),
        )

    def forward(self, cls: torch.Tensor) -> torch.Tensor:
        return self.net(cls)

@torch.no_grad()
def baseline_probs(xb: torch.Tensor) -> torch.Tensor:
    model.eval(); head.eval()
    feats = model.forward_features(xb)
    if isinstance(feats, dict) and "x" in feats:
        feats = feats["x"]
    cls = feats[:, 0, :]
    logits = head(cls)
    return torch.softmax(logits, dim=1)

@torch.no_grad()
def masked_probs(xb: torch.Tensor, G: torch.nn.Module, mode="PP") -> tuple[torch.Tensor, float, float]:
    model.eval(); head.eval(); G.eval()
    feats = model.forward_features(xb)
    if isinstance(feats, dict) and "x" in feats:
        feats = feats["x"]
    cls = feats[:, 0, :]                 # [B, D]
    fmask = torch.sigmoid(G(cls))        # [B, D]
    modified = cls * fmask if mode == "PP" else cls + fmask
    logits = head(modified)
    probs = torch.softmax(logits, dim=1)
    nz_ratio = (fmask.abs() > 1e-6).float().mean().item()
    l1 = fmask.abs().sum(dim=1).mean().item()
    return probs, nz_ratio, l1

def load_generator(path: str, D: int, mode="PP"):
    """Recreate the same MaskGenerator and load its state dict."""
    G = MaskGenerator(d=D, mode=mode)
    sd = torch.load(path, map_location="cpu")
    G.load_state_dict(sd, strict=True)
    return G.to(DEVICE).eval()

# --- Collect trained Gs ---
run_dir = "/kaggle/working/cfe_all"
paths = sorted(glob.glob(os.path.join(run_dir, "G_PP_class_*.pt")))
assert len(paths) > 0, f"No trained G found in {run_dir}/*.pt"
print(f"Found {len(paths)} trained generators.")

# quick test loader (reuse ds_test/dl_test from the data cell)
eval_bs = 64
eval_loader = DataLoader(ds_test, batch_size=eval_bs, shuffle=False, num_workers=2, pin_memory=True)

rows = []
for p in paths:
    m = re.search(r"class_(\d+)\.pt$", p)
    cid = int(m.group(1)) if m else None
    cname = id2label[cid] if cid is not None and cid < len(id2label) else str(cid)
    print(f"\nEvaluating G for class {cid:03d}: {cname}")

    # IMPORTANT: pass D (uppercase) — the CLS dimension you printed in Cell-1
    G = load_generator(p, D, mode="PP")

    total = 0
    sum_base = 0.0
    sum_mask = 0.0
    sum_delta = 0.0
    nz_collect, l1_collect = [], []

    for xb, yb in eval_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        base = baseline_probs(xb)               # [B, C]
        mask, nz_ratio, l1 = masked_probs(xb, G, mode="PP")

        b = xb.size(0)
        base_c  = base[:, cid].mean().item()
        mask_c  = mask[:, cid].mean().item()
        delta_c = (mask[:, cid] - base[:, cid]).mean().item()

        total     += b
        sum_base  += base_c  * b
        sum_mask  += mask_c  * b
        sum_delta += delta_c * b
        nz_collect.append(nz_ratio)
        l1_collect.append(l1)

    base_mean = sum_base / total
    mask_mean = sum_mask / total
    delta_mean = sum_delta / total
    nz_mean = float(np.mean(nz_collect))
    l1_mean = float(np.mean(l1_collect))

    rows.append({
        "class_id": cid,
        "class_name": cname,
        "baseline_p_alter": base_mean,
        "masked_p_alter": mask_mean,
        "delta": mask_mean - base_mean,
        "nz_ratio": nz_mean,
        "l1_mean": l1_mean,
        "generator_path": p,
    })

# Save summary
df = pd.DataFrame(rows).sort_values("delta", ascending=False)
summary_csv = os.path.join(run_dir, "cfe_eval_summary.csv")
df.to_csv(summary_csv, index=False)
print("\nSaved summary:", summary_csv)
display(df.head(10))

# Zip trained G’s + summary
zip_path = os.path.join(run_dir, "cfe_top5_generators.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(summary_csv, arcname=os.path.basename(summary_csv))
    for p in paths:
        z.write(p, arcname=os.path.basename(p))
print("Zipped generators to:", zip_path)

Found 5 trained generators.

Evaluating G for class 017: 018.Spotted Catbird

Evaluating G for class 069: 070.Green Violetear

Evaluating G for class 073: 074.Florida Jay

Evaluating G for class 092: 093.Clark Nutcracker

Evaluating G for class 168: 169.Magnolia Warbler

Saved summary: /kaggle/working/cfe_all/cfe_eval_summary.csv


,class_id,class_name,baseline_p_alter,masked_p_alter,delta,nz_ratio,l1_mean,generator_path
0,17,018.Spotted Catbird,0.003455,0.011326,0.007871,1.0,441.225980,/kaggle/working/cfe_all/G_PP_class_017.pt
1,69,070.Green Violetear,0.006032,0.008212,0.002181,1.0,416.442913,/kaggle/working/cfe_all/G_PP_class_069.pt
3,92,093.Clark Nutcracker,0.005160,0.007319,0.002159,1.0,415.568562,/kaggle/working/cfe_all/G_PP_class_092.pt
2,73,074.Florida Jay,0.005204,0.006930,0.001725,1.0,417.608996,/kaggle/working/cfe_all/G_PP_class_073.pt
4,168,169.Magnolia Warbler,0.005453,0.006665,0.001212,1.0,412.017349,/kaggle/working/cfe_all/G_PP_class_168.pt


Zipped generators to: /kaggle/working/cfe_all/cfe_top5_generators.zip


# Cell 6 — Visualize CFE (PP): per-image montages + CSVs + zip

In [7]:
# Cell 6 — Visualize CFE (PP): per-image montages + CSVs + zip

import os, glob, json, random, zipfile
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ---------- helpers reused (safe re-defs) ----------

def get_cls_from_features(features):
    if isinstance(features, torch.Tensor) and features.ndim == 3:
        return features[:, 0]  # [B, tokens, D]
    if isinstance(features, dict):
        if "x_norm_clstoken" in features:
            return features["x_norm_clstoken"]
        if "pre_logits" in features:
            x = features["pre_logits"]
            return x if x.ndim == 2 else x[:, 0]
    raise RuntimeError("Could not extract CLS embedding from features.")

@torch.no_grad()
def baseline_probs(xb: torch.Tensor) -> torch.Tensor:
    model.eval(); head.eval()
    feats = model.forward_features(xb)
    if isinstance(feats, dict) and "x" in feats:
        feats = feats["x"]
    logits = head(get_cls_from_features(feats))
    return torch.softmax(logits, dim=1)

class MaskGenerator(nn.Module):
    def __init__(self, d: int, mode: str = "PP"):
        super().__init__()
        assert mode in ("PP", "PN")
        self.net = nn.Sequential(
            nn.Linear(d, d), nn.ReLU(inplace=True),
            nn.Linear(d, d),
            nn.Sigmoid() if mode == "PP" else nn.ReLU(inplace=True),
        )
    def forward(self, cls): return self.net(cls)

def load_generator_mask(path: str, d: int) -> nn.Module:
    """Load the trained G (two-layer MLP with sigmoid)."""
    G = MaskGenerator(d, mode="PP").to(DEVICE)
    sd = torch.load(path, map_location="cpu")
    G.load_state_dict(sd, strict=True)
    G.eval()
    return G

@torch.no_grad()
def masked_probs_variant(xb: torch.Tensor, G: nn.Module, binarize: bool=False) -> Tuple[torch.Tensor, torch.Tensor]:
    """Return probs and fmask for one forward; binarize mask for 'visual' pass if requested."""
    model.eval(); head.eval(); G.eval()
    feats = model.forward_features(xb)
    if isinstance(feats, dict) and "x" in feats:
        feats = feats["x"]
    cls = get_cls_from_features(feats)            # [B, D]
    fmask = G(cls)                                # already sigmoid for PP
    if binarize:
        fmask = (fmask > 0.5).float()
    modified = cls * fmask                        # PP
    logits = head(modified)
    probs = torch.softmax(logits, dim=1)
    return probs, fmask

# inverse-norm to show nice images
MEAN = np.array([0.485, 0.456, 0.406]).reshape(1,1,3)
STD  = np.array([0.229, 0.224, 0.225]).reshape(1,1,3)
def to_uint8(img_t: torch.Tensor) -> np.ndarray:
    """CHW (torch) -> HWC uint8 (0..255) with de-normalization."""
    x = img_t.detach().cpu().permute(1,2,0).numpy()
    x = (x * STD + MEAN).clip(0, 1)
    return (x * 255).astype(np.uint8)

# ---------- where to write artifacts ----------
RUN_DIR = "/kaggle/working/cfe_all"
QUAL_DIR = os.path.join(RUN_DIR, "qual")
CSV_DIR  = os.path.join(RUN_DIR, "qual_csv")
os.makedirs(QUAL_DIR, exist_ok=True)
os.makedirs(CSV_DIR,  exist_ok=True)

# discover trained generators
g_paths = sorted(glob.glob(os.path.join(RUN_DIR, "G_PP_class_*.pt")))
assert len(g_paths) > 0, "No trained generators found in /kaggle/working/cfe_all"

# D should already be known from Cell-1/2; if not, infer once:
try:
    D
except NameError:
    with torch.no_grad():
        dummy = torch.randn(1,3,224,224, device=DEVICE)
        feats = model.forward_features(dummy)
        D = get_cls_from_features(feats).shape[-1]
print("Using CLS dim D =", D)

# choose K samples per class (test set, true label == class)
SAMPLES_PER_CLASS = 8
rng = random.Random(42)

def pick_indices_for_class(ds, class_id: int, k: int) -> List[int]:
    idxs = [i for i, y in enumerate(ds.labels) if y == class_id]
    if len(idxs) == 0: return []
    rng.shuffle(idxs)
    return idxs[:min(k, len(idxs))]

# ---------- main loop ----------
summary_rows = []
for p in g_paths:
    cid = int(os.path.basename(p).split("_")[-1].split(".")[0])
    cname = id2label[cid] if cid < len(id2label) else str(cid)
    print(f"\nVisualizing class {cid:03d}: {cname}")

    # load generator
    G = load_generator_mask(p, D)

    # pick samples
    sel = pick_indices_for_class(ds_test, cid, SAMPLES_PER_CLASS)
    if not sel:
        print("  (no test samples found for this class)")
        continue

    # collect per-image stats
    rows = []
    imgs_for_grid = []
    titles = []

    for idx in sel:
        x, y = ds_test[idx]            # transformed tensor
        path = ds_test.paths[idx]
        xb = x.unsqueeze(0).to(DEVICE)

        base = baseline_probs(xb)                              # [1, C]
        prob_base = float(base[0, cid].item())

        prob_soft, fmask_soft = masked_probs_variant(xb, G, binarize=False)
        prob_bin,  fmask_bin  = masked_probs_variant(xb, G, binarize=True)

        prob_soft_c = float(prob_soft[0, cid].item())
        prob_bin_c  = float(prob_bin[0,  cid].item())

        pred_before = int(base.argmax(dim=1).item())
        pred_after_soft = int(prob_soft.argmax(dim=1).item())
        pred_after_bin  = int(prob_bin.argmax(dim=1).item())

        rows.append({
            "img_path": path,
            "class_id": cid,
            "class_name": cname,
            "base_prob": prob_base,
            "masked_prob_soft": prob_soft_c,
            "masked_prob_bin":  prob_bin_c,
            "delta_soft": prob_soft_c - prob_base,
            "delta_bin":  prob_bin_c  - prob_base,
            "pred_before": pred_before,
            "pred_after_soft": pred_after_soft,
            "pred_after_bin":  pred_after_bin,
        })

        # for montage
        imgs_for_grid.append(to_uint8(x))
        titles.append(f"b:{prob_base:.3g}\n s:{prob_soft_c:.3g} Δ:{(prob_soft_c-prob_base):+.3g}")

    # save CSV for this class
    df_cls = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, f"class_{cid:03d}.csv")
    df_cls.to_csv(csv_path, index=False)
    print("  CSV ->", csv_path)

    # montage (H x W grid)
    cols = 4
    rows_grid = int(np.ceil(len(imgs_for_grid) / cols))
    fig_w = 4 * cols
    fig_h = 4 * rows_grid
    plt.figure(figsize=(fig_w, fig_h))
    for i, img in enumerate(imgs_for_grid):
        ax = plt.subplot(rows_grid, cols, i+1)
        ax.imshow(img)
        ax.set_title(titles[i], fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Class {cid:03d} — {cname}  (soft mask shown in stats; grid shows images only)", fontsize=12)
    png_path = os.path.join(QUAL_DIR, f"class_{cid:03d}.png")
    plt.tight_layout()
    plt.savefig(png_path, bbox_inches="tight", dpi=150)
    plt.close()
    print("  PNG ->", png_path)

    # aggregate for summary
    mean_delta = float(df_cls["delta_soft"].mean())
    summary_rows.append({"class_id": cid, "class_name": cname, "mean_delta_soft": mean_delta,
                         "n_images": len(df_cls)})

# summary table + quick plot
df_sum = pd.DataFrame(summary_rows).sort_values("mean_delta_soft", ascending=False)
sum_csv = os.path.join(RUN_DIR, "cfe_visual_summary.csv")
df_sum.to_csv(sum_csv, index=False)
print("\nSaved visual summary:", sum_csv)
display(df_sum)

# zip everything for convenience
zip_path = os.path.join(RUN_DIR, "cfe_visuals.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for root in (QUAL_DIR, CSV_DIR):
        for r, _, files in os.walk(root):
            for f in files:
                fp = os.path.join(r, f)
                z.write(fp, arcname=os.path.relpath(fp, RUN_DIR))
    z.write(sum_csv, arcname=os.path.basename(sum_csv))
print("Zipped visuals to:", zip_path)

Using CLS dim D = 768

Visualizing class 017: 018.Spotted Catbird
  CSV -> /kaggle/working/cfe_all/qual_csv/class_017.csv
  PNG -> /kaggle/working/cfe_all/qual/class_017.png

Visualizing class 069: 070.Green Violetear
  CSV -> /kaggle/working/cfe_all/qual_csv/class_069.csv
  PNG -> /kaggle/working/cfe_all/qual/class_069.png

Visualizing class 073: 074.Florida Jay
  CSV -> /kaggle/working/cfe_all/qual_csv/class_073.csv
  PNG -> /kaggle/working/cfe_all/qual/class_073.png

Visualizing class 092: 093.Clark Nutcracker
  CSV -> /kaggle/working/cfe_all/qual_csv/class_092.csv
  PNG -> /kaggle/working/cfe_all/qual/class_092.png

Visualizing class 168: 169.Magnolia Warbler
  CSV -> /kaggle/working/cfe_all/qual_csv/class_168.csv
  PNG -> /kaggle/working/cfe_all/qual/class_168.png

Saved visual summary: /kaggle/working/cfe_all/cfe_visual_summary.csv


,class_id,class_name,mean_delta_soft,n_images
0,17,018.Spotted Catbird,-0.000581,8
3,92,093.Clark Nutcracker,-0.600732,8
2,73,074.Florida Jay,-0.653479,8
1,69,070.Green Violetear,-0.663659,8
4,168,169.Magnolia Warbler,-0.703437,8


Zipped visuals to: /kaggle/working/cfe_all/cfe_visuals.zip
